Notebook: 06_qtc_methods.ipynb

Purpose: Calculate standard QTc formulas at the beat level.

Inputs:
- qt_measurements.parquet

Outputs:
- qtc_comparison.parquet

# 06 — QTc Calculations

Compute Fridericia, Bazett, Framingham, and Hodges corrections for each beat and summarize formula spread.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
sys.path.insert(0, str(Path.cwd().parent / 'src'))
from ecg_analytics.qtc.formulas import compute_all_qtc

root_dir = Path.cwd().parent
artifacts_dir = root_dir / 'artifacts'
qt_path = artifacts_dir / 'qt_measurements.parquet'
qt_df = pd.read_parquet(qt_path)
qt_df['qt_ms'] = pd.to_numeric(qt_df['qt_ms'], errors='coerce')
qt_df['rr_ms'] = pd.to_numeric(qt_df['rr_ms'], errors='coerce')

records = []
for (record_id, beat_id), subset in qt_df.groupby(['record_id','beat_id']):
    valid = subset.dropna(subset=['qt_ms','rr_ms'])
    if valid.empty:
        continue
    qt_ms = float(valid['qt_ms'].mean())
    rr_ms = float(valid['rr_ms'].mean())
    qtc = compute_all_qtc(qt_ms, rr_ms)
    qtc_values = np.array([qtc['fridericia'], qtc['bazett'], qtc['framingham'], qtc['hodges']], dtype=float)
    records.append({
        'record_id': record_id,
        'beat_id': int(beat_id),
        'rr_ms': rr_ms,
        'qtc_bazett': float(qtc['bazett']),
        'qtc_fridericia': float(qtc['fridericia']),
        'qtc_framingham': float(qtc['framingham']),
        'qtc_hodges': float(qtc['hodges']),
        'qtc_formula_variance': float(np.nanvar(qtc_values)),
        'qtc_formula_bias': float(np.nanmax(qtc_values) - np.nanmin(qtc_values)),
    })

qtc_comparison = pd.DataFrame(records)
expected = {'record_id','beat_id','rr_ms','qtc_bazett','qtc_fridericia','qtc_framingham','qtc_hodges','qtc_formula_variance','qtc_formula_bias'}
assert expected.issubset(set(qtc_comparison.columns)), 'qtc_comparison schema mismatch'
assert qtc_comparison[['record_id','beat_id']].duplicated().sum() == 0

qtc_comparison.to_parquet(artifacts_dir / 'qtc_comparison.parquet', index=False)
print('Wrote qtc_comparison.parquet')
